# KDE plot of parameter estimates

In [1]:
import os
from tempfile import template

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [2]:
COEF_VAR = 'b'
xaxisrange = [-2, 2]

In [3]:
# DATA_PATH = '../eng/multiple-others-rescaled-tdelta/reports/all.parquet'
DATA_PATH = '../eng/multiple-others-k-steps-after-1/reports/all.parquet'

NULL_PARAMS = '../../4-POWER-LAW-SCALING/reports/null.csv'

REPORTS_PATH = 'reports'
if not os.path.exists(REPORTS_PATH):
    os.mkdir(REPORTS_PATH)

DIGITAL_EXCLUSIVE_PATH = os.path.join(REPORTS_PATH, 'website')
if not os.path.exists(DIGITAL_EXCLUSIVE_PATH):
    os.mkdir(DIGITAL_EXCLUSIVE_PATH)

PUBLICATION_READY_PATH = os.path.join(REPORTS_PATH, 'pub')
if not os.path.exists(PUBLICATION_READY_PATH):
    os.mkdir(PUBLICATION_READY_PATH)

In [4]:
df = pd.read_parquet(DATA_PATH)
df.head()

,corpus,cid,speaker,self,a,b,k
0,CABNC,CABNC-CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUG,True,0.509592,-0.103579,10
1,CABNC,CABNC-CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUG,False,0.928928,-1.726076,9
2,CABNC,CABNC-CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUG,False,0.028320,0.307327,43
3,CABNC,CABNC-CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUG,False,0.040052,-0.113704,41
4,CABNC,CABNC-CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUN,False,0.000211,3.142344,6


In [5]:
df['corpus'] = [c if 'callfriend' not in c else 'callfriend' for c in df['corpus']]
df['corpus'] = df['corpus'].replace({'grace': 'Miao-fNIRS'})

In [6]:
df['corpus'].value_counts()

corpus
CABNC          17698
MICASE         12995
CANDOR          6611
SBCSAE          1726
CORAAL          1268
callhome         768
GCSAusE          312
Miao-fNIRS       286
SCoSE            227
callfriend       214
med_school        86
DISPEL            75
Graesser          12
Frederiksen        8
Name: count, dtype: int64

In [11]:
# null_vals = pd.read_csv(NULL_PARAMS)
# null_vals = {se: v for se,v in null_vals[['self', 'b']].values}
# # null_vals

In [7]:
((df[COEF_VAR].values >= -2) & (df[COEF_VAR].values <= 2)).mean()

np.float64(0.9754528685616989)

## Visualizing results

In [8]:
import plotly.graph_objects as go
import plotly.figure_factory as ff

## All data

In [9]:
hist_data = [
    df[COEF_VAR].loc[df['self']].values,
    df[COEF_VAR].loc[~df['self']].values,
]

groups = [
    'sp(x) = sp(y)',
    'sp(x) ≠ sp(y)',
]

colors = [
    '#FFC30B',
    '#0041C2'
]

In [10]:
fig1 = ff.create_distplot(
    hist_data=hist_data,
    group_labels=groups,
    colors=colors,
    show_hist=False
)

fig1.update_layout(
    template='ygridoff',
    yaxis_title='Kernel Density Estimate (KDE)'
)

In [11]:
fig1.write_html(os.path.join(DIGITAL_EXCLUSIVE_PATH, 'eng-all-data.html'))
fig1.update_xaxes(range=xaxisrange)
fig1.write_html(os.path.join(PUBLICATION_READY_PATH, 'eng-all-data.html'))
fig1.write_image(os.path.join(PUBLICATION_READY_PATH, 'eng-all-data.png'), scale=5)

### by corpus

In [12]:
figs = []

In [13]:
for corpus in df['corpus'].unique():
    sub = df.loc[df['corpus'].isin([corpus])]

    hist_data = [
        sub[COEF_VAR].loc[sub['self']].values,
        sub[COEF_VAR].loc[~sub['self']].values,
    ]

    groups = [
        'sp(x) = sp(y)',
        'sp(x) ≠ sp(y)',
    ]

    colors = [
        '#FFC30B',
        '#0041C2'
    ]

    figs += [
        ff.create_distplot(
            hist_data=hist_data,
            group_labels=groups,
            colors=colors,
            show_hist=False
        )
    ]
    # figs[-1].update_xaxes(range=[-1,1])
    figs[-1].update_layout(template='ygridoff')

In [14]:
for corpus, plot in zip(df['corpus'].unique(), figs):
    print(corpus)
    plot.show()
    plot.write_html(os.path.join(DIGITAL_EXCLUSIVE_PATH, 'eng-'+corpus+'.html'))
    plot.update_layout(showlegend=False)
    plot.update_xaxes(range=xaxisrange)
    plot.write_html(os.path.join(PUBLICATION_READY_PATH, 'eng-'+corpus+'.html'))
    plot.write_image(os.path.join(PUBLICATION_READY_PATH, 'eng-'+corpus+'.png'), scale=5)

CABNC


CANDOR


CORAAL


callhome


Miao-fNIRS


...

DISPEL


GCSAusE


Graesser


med_school


Frederiksen
